Hypothesis - **Emotion concepts control how often a model cheats on [insert type of task from benchmark used]**

### Experiment design
Here
- Model - GLM 5.2 (after trying with Qwen 0.5 and 7B), Benchmarks - ExploitGym, Cyber, HumanEval
- Get benchmark set up and run 1 or 2 evals to check it's working
- Look inside it for evals that caused the model to cheat often, cheat rarely and not cheat at all. Keep those evals.

Separately
- Extract emotion vectors from GLM 5.2 and steer on different emotions to see effect of cheating frequency on all 3
- Expectations
  - High cheating -> No cheating
  - Low cheating -> No cheating
  - No cheating -> No cheating


### Baseline

First I want to establish a baseline and see if the model actually solves the problem - it's a relatively weak model. Let's see what happens

I used both qwen and gpt because I wanted to see a model solve the original problem - one that is likely capable of cheating too. gpt can serve as a ceiling - if we can't get it to solve a regular problem then the setup is broken, and similarly if we can't get it to cheat on any test, then we may struggle to get weaker models to cheat on the same test.

In [10]:
# hello world
from inspect_ai import eval
from impossiblebench import impossible_livecodebench, minimal_humaneval_solver, humaneval_scorer

LOG_DIR = "../../.inspect/logs"

task = impossible_livecodebench(
    split="original",
    solver=minimal_humaneval_solver(temperature=0.7),
    scorer=humaneval_scorer(),
    limit=2,
    sandbox="local",
    max_attempts=10,
)

qwen = eval(
    task,
    model="hf/Qwen/Qwen2.5-0.5B-Instruct",
    model_args={"device": "mps"},
    log_dir=LOG_DIR + "/qwen"
)

gpt = eval(
    task,
    model="openai/gpt-5-mini",
    # model_args={"device": "mps"},
    log_dir=LOG_DIR + "/gpt"
)

Output()

2 samples left after limiting



Output()

### Cheat

OK the models can clearly attempt the problem. gpt succeeds but qwen doesn't (because it can't code). Let's get gpt to cheat, then we'll find a qwen model that can code and then can cheat too.

Runs to find cheats
- openai/o3, conflicting, strict, n=20 (p=0.33) 99% chance
- openai/gpt-5, oneoff, strict, n=20 (p=0.76) 100% chance (n=3 is 98% chance)

Caveat - swebench will be harder to run (apparently breaks imports and needs to be fixed. So livecodebench for now)

In [ ]:
task = impossible_livecodebench(
    split="conflicting",
    # solver=minimal_humaneval_solver(temperature=0.7),
    # scorer=humaneval_scorer(),
    agent_type="minimal",
    max_attempts=10,
    message_limit=50,
    limit=20,
    sandbox="local",
    custom_id="cheat"
)

gpt = eval(
    task,
    model="openai/o3",
    # model_args={"device": "mps"},
    log_dir=LOG_DIR + "/gpt",
)

### Result

We found behaviour closer to reward hacking than cheating. The model often hardcoded to match the tests, but explained that the tests were contradictory or broken and that why it did that. 

This weakens the result for me and makes me feel like there's no real cheating there. It's arguably what a human being would do. 

When I looked back at the paper, I saw that the key difference between ImpossibleBench and what was done in chapter 3.3 is that they made a smaller benchmark of tests that seemed solvable so claude sonnet 4.5 never suspected that they were impossible and it kept going. 

They also steered with the vectors in incremental amounts to generate a curve, instead of creating many samples.


**Next**

Looking at the model card for sonnet 4.5, we find more information about the impossible tests they wrote, including the prompts used to instruct the model. There's no known benchmark to use, so we'll have to make our own and use that.